# AI Workout Planner Agent

**Goal:** Build an AI agent that recommends and tracks workouts based on the user's fitness goal and previous workout sessions.

**Tools:** `suggest_exercise(goal)`, `log_session(exercise)`

**Memory:** Stores previous workout sessions to provide dynamic recommendations.

This project demonstrates a simple agentic workflow where the agent understands the user's request, selects the appropriate tool, and uses previous workout history while making recommendations.

## 1. Agent Configuration

This section configures the AI model used by the Workout Planner Agent and initializes the environment required for agent execution.

In [1]:
from cse476.lanes import get_client, MODEL, describe

# WHY: print this every time. Half of all "it stopped working" reports are
# actually "I am on a different lane than I thought".
print(describe())
client = get_client()

Lane: Microsoft Foundry (foundry, billed)  |  Model: chat-demo


## 2. Agent Tools and Memory

The agent uses predefined tools to suggest exercises and log completed workout sessions.

`SESSION_HISTORY` stores previous workout sessions and acts as the agent's memory. The `suggest_exercise` tool uses this history to provide recommendations that are not simply repeated.

In [2]:
from collections import Counter

EXERCISE_LIBRARY = {
    "strength": [
        {"name": "Barbell Squats", "muscle_group": "legs"},
        {"name": "Bench Press", "muscle_group": "chest"},
        {"name": "Deadlifts", "muscle_group": "back"},
        {"name": "Overhead Press", "muscle_group": "shoulders"},
        {"name": "Pull-ups", "muscle_group": "back"},
    ],
    "cardio": [
        {"name": "Running Intervals", "muscle_group": "cardio"},
        {"name": "Jump Rope", "muscle_group": "cardio"},
        {"name": "Cycling", "muscle_group": "cardio"},
        {"name": "Rowing Machine", "muscle_group": "cardio"},
    ],
    "flexibility": [
        {"name": "Yoga Flow", "muscle_group": "mobility"},
        {"name": "Dynamic Stretching", "muscle_group": "mobility"},
        {"name": "Foam Rolling", "muscle_group": "mobility"},
    ],
}

# Memory: each entry is one completed session.
SESSION_HISTORY = []


def suggest_exercise(goal: str) -> str:
    goal = goal.lower().strip()
    if goal not in EXERCISE_LIBRARY:
        return f"Unknown goal '{goal}'. Available goals: {list(EXERCISE_LIBRARY)}"

    candidates = EXERCISE_LIBRARY[goal]

    # WHY: this is the "agentic, not a fixed plan" part. Look at the last two
    # sessions and steer away from what was just trained.
    recent = SESSION_HISTORY[-2:]
    recent_muscle_groups = {s["muscle_group"] for s in recent}
    recent_exercises = {s["exercise"] for s in recent}

    fresh = [
        c for c in candidates
        if c["muscle_group"] not in recent_muscle_groups and c["name"] not in recent_exercises
    ]
    pool = fresh if fresh else candidates  # everything was hit recently, fall back

    # Among the fresh options, prefer whatever has been done least overall,
    # so the rotation doesn't just ping-pong between two exercises forever.
    counts = Counter(s["exercise"] for s in SESSION_HISTORY)
    pick = min(pool, key=lambda c: counts.get(c["name"], 0))

    note = "" if fresh else " (repeating a muscle group — every option was used in your last 2 sessions)"
    return f"Suggested: {pick['name']} (muscle group: {pick['muscle_group']}) for goal '{goal}'{note}."


def log_session(exercise: str) -> str:
    for goal, exs in EXERCISE_LIBRARY.items():
        for e in exs:
            if e["name"].lower() == exercise.lower():
                SESSION_HISTORY.append({"exercise": e["name"], "muscle_group": e["muscle_group"], "goal": goal})
                return f"Logged '{e['name']}' (session #{len(SESSION_HISTORY)})."
    # Allow logging something outside the library too, just without rotation logic.
    SESSION_HISTORY.append({"exercise": exercise, "muscle_group": "unknown", "goal": "unknown"})
    return f"Logged '{exercise}' (session #{len(SESSION_HISTORY)}). Not in the library, so it won't affect future rotation."


# Test tools before wiring them into an agent.
print(suggest_exercise("strength"))
print(log_session("Barbell Squats"))
print(suggest_exercise("strength"))

Suggested: Barbell Squats (muscle group: legs) for goal 'strength'.
Logged 'Barbell Squats' (session #1).
Suggested: Bench Press (muscle group: chest) for goal 'strength'.


### Tool Registry

The registry defines the tools that the agent is allowed to execute. Only registered functions can be called by the agent.

In [3]:
REGISTRY = {
    "suggest_exercise": suggest_exercise,
    "log_session": log_session,
}

### Tool Schema

The tool schema provides the AI model with information about the available tools, their purpose, and the required input parameters.

In [4]:
TOOL_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "suggest_exercise",
            "description": (
                "Suggest the next exercise for a workout goal. This already accounts "
                "for the user's recent session history to avoid repeating the same "
                "muscle group back-to-back, so do not invent your own fixed weekly "
                "plan — always call this to decide what's next."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "goal": {
                        "type": "string",
                        "description": "Workout goal.",
                        "enum": ["strength", "cardio", "flexibility"],
                    },
                },
                "required": ["goal"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_session",
            "description": (
                "Record that the user completed a specific exercise. Use this when "
                "the user confirms they finished a workout, so future suggestions "
                "account for it."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "exercise": {
                        "type": "string",
                        "description": "Exact exercise name, e.g. 'Barbell Squats'.",
                    },
                },
                "required": ["exercise"],
            },
        },
    },
]

## 3. Agent Workflow

The agent follows a Think → Act → Observe workflow.

It understands the user's request, selects an appropriate tool, executes the tool, observes the result, and generates a response. The workflow is bounded by `max_steps` to prevent unnecessary loops.

In [5]:
import json

SYSTEM = (
    "You are a workout planning assistant. You have two tools: suggest_exercise, "
    "which recommends the next exercise for a goal while already accounting for "
    "the user's recent sessions, and log_session, which records a completed "
    "exercise. Do not design your own fixed weekly schedule — always call "
    "suggest_exercise to decide what's next, and call log_session whenever the "
    "user says they finished something. When you have everything you need, answer "
    "the user directly and stop calling tools."
)


def run_agent(goal: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": goal},
    ]

    for step in range(1, max_steps + 1):
        # THINK
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMA
        )
        message = response.choices[0].message

        # WHY: the assistant turn goes back in whether or not it asked for a tool.
        messages.append(message.model_dump(exclude_none=True))

        # EXIT 1: the model stopped asking for tools.
        if not message.tool_calls:
            if verbose:
                print(f"[step {step}] done")
            return message.content or ""

        # ACT
        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")

            if name in REGISTRY:
                result = REGISTRY[name](**args)   # <-- the only executing line
            else:
                result = f"Error: no tool named '{name}'. Available: {list(REGISTRY)}"

            if verbose:
                print(f"[step {step}] {name}({args}) -> {result}")

            # OBSERVE
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    # EXIT 2: the budget ran out.
    return f"Stopped after {max_steps} steps without reaching a final answer."


print("agent defined")

agent defined


## 4. Testing the Workout Planner Agent

This section tests the agent by requesting workout recommendations and logging completed exercises.

The results demonstrate how previous workout sessions influence future recommendations.

In [6]:
SESSION_HISTORY.clear()

print(run_agent("I want a strength workout today. What should I do?"))

[step 1] suggest_exercise({'goal': 'strength'}) -> Suggested: Barbell Squats (muscle group: legs) for goal 'strength'.
[step 2] done
For your strength workout today, I suggest doing Barbell Squats. Let me know when you have finished, and I can log it for you.


In [7]:
print(run_agent("I just finished the Barbell Squats, log that session."))

[step 1] log_session({'exercise': 'Barbell Squats'}) -> Logged 'Barbell Squats' (session #1).
[step 2] suggest_exercise({'goal': 'strength'}) -> Suggested: Bench Press (muscle group: chest) for goal 'strength'.
[step 3] done
Great job on completing the Barbell Squats! For your next strength workout, I suggest doing Bench Press which focuses on your chest. Let me know when you finish it.


In [8]:
# Same request as before. Because Barbell Squats (legs) was just logged,
# suggest_exercise should steer to a different muscle group this time.
print(run_agent("Give me another strength workout."))

[step 1] suggest_exercise({'goal': 'strength'}) -> Suggested: Bench Press (muscle group: chest) for goal 'strength'.
[step 2] done
I suggest doing Bench Press for your next strength workout. Let me know when you finish, and I can help you plan the next session!


In [9]:
print(run_agent("Done, I did the Bench Press. Log it, and then tell me what's next.", max_steps=8))

[step 1] log_session({'exercise': 'Bench Press'}) -> Logged 'Bench Press' (session #2).
[step 1] suggest_exercise({'goal': 'strength'}) -> Suggested: Deadlifts (muscle group: back) for goal 'strength'.
[step 2] done
I've logged your Bench Press session. For your next strength workout, I suggest doing Deadlifts to work on your back muscles. Let me know when you finish or if you need anything else!


In [10]:
# Inspect memory directly (not something the model can see — this is just us
# checking the agent's state from the outside).
SESSION_HISTORY

[{'exercise': 'Barbell Squats', 'muscle_group': 'legs', 'goal': 'strength'},
 {'exercise': 'Bench Press', 'muscle_group': 'chest', 'goal': 'strength'}]

## 5. Error Handling

This section tests how the agent handles unsupported workout goals and invalid requests.

The agent uses bounded execution with `max_steps` to ensure that the workflow does not continue indefinitely.

In [11]:
print(run_agent("Give me a powerlifting-specific plan for tomorrow.", max_steps=6))

[step 1] suggest_exercise({'goal': 'strength'}) -> Suggested: Deadlifts (muscle group: back) for goal 'strength'.
[step 2] done
For your powerlifting-specific plan for tomorrow, I suggest starting with deadlifts. Let me know when you've completed this exercise or if you want additional exercises included in the plan.


## 6. Future Improvements

The current project can be extended by adding persistent memory, workout dates, additional exercises, and more agent tools.

Possible improvements include storing workout history in a database and providing more personalized workout recommendations.